In [8]:
from typing import Dict, Any
from pathlib import Path

from adabn.utils import get_bn_layers_ordered

from model_ranking import (
    SelfTrainingModelConfig
)
from pytorch3dunet.unet3d.model import (
    get_model,  # pyright: ignore[reportUnknownVariableType]
)
from pytorch3dunet.unet3d.utils import (
    load_checkpoint,  # pyright: ignore[reportUnknownVariableType]
)


In [9]:
config: Dict[str, Any] = {
    "model": {
        "name": "UNet2d_as3d",
        "in_channels": 1,
        "out_channels": 1,
        "layer_order": "bcr",
        "f_maps": 32,
        "final_sigmoid": True,
        "feature_return": False,
        "is_segmentation": True,
        "feature_perturbation": None
    },
    "source_checkpoint": "/g/kreshuk/talks/segmentation_ModelSelection/experiments/Hmito/BatchNorm/Hm_model4/best_checkpoint.pytorch"
}
model_cfg = SelfTrainingModelConfig.model_validate(config)


In [10]:
model = get_model(model_cfg.model.model_dump())
print("Initialize from source model:", model_cfg.source_checkpoint)
src_ckpt_path = model_cfg.source_checkpoint
assert src_ckpt_path is not None, "Source checkpoint must be provided"
if Path(src_ckpt_path).suffix == ".pt":
    model_key = "model_state"
else:
    model_key = "model_state_dict"
_ = load_checkpoint(src_ckpt_path, model, model_key=model_key)

Initialize from source model: /g/kreshuk/talks/segmentation_ModelSelection/experiments/Hmito/BatchNorm/Hm_model4/best_checkpoint.pytorch


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:64: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(checkpoint_path, map_location="cpu")


In [11]:
config2: Dict[str, Any] = {
    "model": {
        "name": "UNet2d_as3d",
        "in_channels": 1,
        "out_channels": 1,
        "layer_order": "bcr",
        "f_maps": 32,
        "final_sigmoid": True,
        "feature_return": False,
        "is_segmentation": True,
        "feature_perturbation": None
    },
    "source_checkpoint": "/g/kreshuk/talks/model_ranking/notebooks/checks/AdaptiveBatchNorm/sequential_adbn/Hmito_to_EPFL_gap/HmtoE_model4/checkpoints/best_checkpoint.pytorch"
}
model_cfg2 = SelfTrainingModelConfig.model_validate(config2)

In [12]:
model2 = get_model(model_cfg.model.model_dump())
print("Initialize from source model:", model_cfg2.source_checkpoint)
src_ckpt_path = model_cfg2.source_checkpoint
assert src_ckpt_path is not None, "Source checkpoint must be provided"
if Path(src_ckpt_path).suffix == ".pt":
    model_key = "model_state"
else:
    model_key = "model_state_dict"
_ = load_checkpoint(src_ckpt_path, model2, model_key=model_key)

Initialize from source model: /g/kreshuk/talks/model_ranking/notebooks/checks/AdaptiveBatchNorm/sequential_adbn/Hmito_to_EPFL_gap/HmtoE_model4/checkpoints/best_checkpoint.pytorch


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:64: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(checkpoint_path, map_location="cpu")


In [17]:
bn_layers = get_bn_layers_ordered(model)
print(len(bn_layers))
print(bn_layers[0])

14
('encoders.0.basic_module.SingleConv1.batchnorm', BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True))


In [19]:
bn_layer_names = [n for n, _ in bn_layers]
print(bn_layer_names[:2])

['encoders.0.basic_module.SingleConv1.batchnorm', 'encoders.0.basic_module.SingleConv2.batchnorm']


In [23]:
for (name, module), (_, modules2) in zip(model.named_modules(), model2.named_modules()):
    if name in bn_layer_names:
        print(60 * "-")
        print(module.running_mean)
        print(modules2.running_mean)

------------------------------------------------------------
tensor([0.1770])
tensor([0.0761])
------------------------------------------------------------
tensor([0.1004, 0.3447, 0.2422, 0.1563, 0.1731, 0.1171, 0.1751, 0.1971, 0.2262,
        0.3939, 0.2814, 0.2125, 0.1252, 0.1155, 0.1238, 0.1635])
tensor([0.0897, 0.3918, 0.2732, 0.1546, 0.1781, 0.1223, 0.1878, 0.2225, 0.2486,
        0.4558, 0.3236, 0.2340, 0.1190, 0.1057, 0.1171, 0.1592])
------------------------------------------------------------
tensor([0.5016, 0.4640, 0.7246, 0.4470, 0.3976, 0.5022, 0.5320, 0.5471, 0.4310,
        0.4047, 0.4620, 0.5176, 0.3696, 0.4945, 0.5864, 0.5134, 0.4417, 0.3863,
        0.4619, 0.6496, 0.6008, 0.4435, 0.5076, 0.4932, 0.5967, 0.4856, 0.5706,
        0.3871, 0.3493, 0.5439, 0.4625, 0.4192])
tensor([0.5802, 0.5184, 0.8113, 0.4732, 0.3987, 0.4774, 0.5247, 0.6484, 0.3850,
        0.3748, 0.4352, 0.5560, 0.3565, 0.5786, 0.6305, 0.4973, 0.4011, 0.3463,
        0.5686, 0.6871, 0.7142, 0.4576, 0.54